# IMPORT LIBRARY

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler # tambahan library dari AE untuk normalisasi rating
import joblib

pd.set_option('display.max_columns', None)

# LOAD SAMPLE DATASET

In [2]:
df = pd.read_csv('sample_data.csv')

# MENAMPILKAN 5 DATA PERTAMA

In [3]:
df.head()

,user_id,product_id,rating,timestamp
0,A3CJGOATIQ948P,B001BYASQ0,3.0,1318550400
1,A1PSUH0U1FPQ6R,B002QXZPFE,4.0,1376870400
2,A23QSTB241NRF3,B0040HJOO2,1.0,1335225600
3,A1IU4JZFDZA9HJ,B004LRO7FW,5.0,1326326400
4,A1B2D4J8KF4DFN,B006DKEQL0,2.0,1368144000


# INFORMASI DATASET

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   user_id     200000 non-null  object 
 1   product_id  200000 non-null  object 
 2   rating      200000 non-null  float64
 3   timestamp   200000 non-null  int64  
dtypes: float64(1), int64(1), object(2)
memory usage: 6.1+ MB


# UKURAN DATASET AWAL

In [5]:
print("Ukuran dataset awal:")
print(df.shape)

Ukuran dataset awal:
(200000, 4)


# CEK MISSING VALUE

In [6]:
print("Jumlah missing value tiap kolom:")
print(df.isnull().sum())

Jumlah missing value tiap kolom:
user_id       0
product_id    0
rating        0
timestamp     0
dtype: int64


# CEK DUPLICATE DATA

In [7]:
duplicate_count = df.duplicated().sum()

print(f"Jumlah data duplikat sebelum cleaning: {duplicate_count}")

Jumlah data duplikat sebelum cleaning: 0


# CEK DISTRIBUSI RATING

In [8]:
print("Distribusi rating:")
print(df['rating'].value_counts().sort_index())

Distribusi rating:
rating
1.0     10817
2.0      9132
3.0     17972
4.0     44681
5.0    117398
Name: count, dtype: int64


# FILTER USER AKTIF

In [9]:
# Hanya mengambil user yang memberi minimal 2 rating

user_counts = df['user_id'].value_counts()

active_users = user_counts[user_counts >= 2].index

df = df[df['user_id'].isin(active_users)]

# UKURAN DATASET SETELAH FILTER USER

In [10]:
print("Shape setelah filtering user:")
print(df.shape)

Shape setelah filtering user:
(199998, 4)


# FILTER PRODUK POPULER

In [11]:
# Hanya mengambil produk dengan minimal 2 interaksi

product_counts = df['product_id'].value_counts()

popular_products = product_counts[product_counts >= 2].index

df = df[df['product_id'].isin(popular_products)]

# UKURAN DATASET SETELAH FILTERING

In [12]:
print("Ukuran dataset setelah filtering:")
print(df.shape)

Ukuran dataset setelah filtering:
(155938, 4)


# CEK JUMLAH USER DAN PRODUK UNIK

In [13]:
unique_users = df['user_id'].nunique()

unique_products = df['product_id'].nunique()

print(f"Jumlah user unik: {unique_users}")
print(f"Jumlah produk unik: {unique_products}")

Jumlah user unik: 13345
Jumlah produk unik: 27447


# CEK SPARSITY DATA

In [14]:
num_interactions = len(df)

sparsity = (
    1 -
    (
        num_interactions /
        (unique_users * unique_products)
    )
) * 100

print(f"Sparsity dataset: {sparsity:.2f}%")

Sparsity dataset: 99.96%


# MENGUBAH TIPE DATA TIMESTAMP

In [15]:
df['timestamp'] = pd.to_datetime(
    df['timestamp'],
    unit='s'
)

# MENAMPILKAN DATASET SETELAH CLEANING

In [16]:
df.head()

,user_id,product_id,rating,timestamp
1,A1PSUH0U1FPQ6R,B002QXZPFE,4.0,2013-08-19
2,A23QSTB241NRF3,B0040HJOO2,1.0,2012-04-24
3,A1IU4JZFDZA9HJ,B004LRO7FW,5.0,2012-01-12
4,A1B2D4J8KF4DFN,B006DKEQL0,2.0,2013-05-10
6,A2SYLJAZO4SPA0,B00006G33N,4.0,2006-08-22


# CEK INFO DATASET SETELAH CLEANING

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 155938 entries, 1 to 199999
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     155938 non-null  object        
 1   product_id  155938 non-null  object        
 2   rating      155938 non-null  float64       
 3   timestamp   155938 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), object(2)
memory usage: 5.9+ MB


# FEATURE SCALING: NORMALISASI RATING

**Tujuan:**
Menskalakan nilai pada kolom `rating` dari rentang asli **(1.0 - 5.0)** menjadi rentang **(0.0 - 1.0)** menggunakan `MinMaxScaler`.

**Alasan Teknis:**
Langkah ini krusial untuk arsitektur *Neural Collaborative Filtering* (NCF) pada tahap *modeling* nanti. Lapisan *output* model AI menggunakan fungsi aktivasi **Sigmoid** (yang hanya bisa mengeluarkan nilai 0 hingga 1). Normalisasi ini juga merupakan strategi utama agar model dapat mencapai metrik evaluasi **MAE < 0.02**

In [18]:
# Inisialisasi MinMaxScaler
scaler = MinMaxScaler()

# Melakukan fit dan transform pada kolom 'rating'
df['rating'] = scaler.fit_transform(df[['rating']])

# Menyimpan scaler ke dalam file .pkl agar bisa digunakan nanti di API dan Dashboard Streamlit
joblib.dump(scaler, 'rating_scaler.pkl')

print("Distribusi rating setelah di-scaling (Rentang 0.0 - 1.0):")
print(df['rating'].describe())
print("\nObjek scaler berhasil disimpan sebagai 'rating_scaler.pkl'")

Distribusi rating setelah di-scaling (Rentang 0.0 - 1.0):
count    155938.000000
mean          0.818853
std           0.274893
min           0.000000
25%           0.750000
50%           1.000000
75%           1.000000
max           1.000000
Name: rating, dtype: float64

Objek scaler berhasil disimpan sebagai 'rating_scaler.pkl'


# SIMULASI INVERSE TRANSFORM (Persiapan Dashboard)

**Tujuan:**
Model AI akan mengeluarkan hasil prediksi rekomendasi dalam bentuk angka **0.0 - 1.0**. Namun, untuk ditampilkan kepada pengguna di layar Dashboard (Front-End), nilai tersebut harus diubah kembali (*inverse*) ke bentuk bintang **1.0 - 5.0** agar logis dan mudah dipahami.

In [19]:
# Load scaler
loaded_scaler = joblib.load('rating_scaler.pkl')

# Simulasi keluaran prediksi dari model AI
prediksi_dummy = np.array([[0.75]])

# Mengembalikan skor 0.75 ke skala rating asli (1 - 5)
rating_dikembalikan = loaded_scaler.inverse_transform(prediksi_dummy)

print(f"Contoh Skor Output Model AI (Scaled) : {prediksi_dummy[0][0]}")
print(f"Skor Dikembalikan ke Skala Asli (1-5)  : {rating_dikembalikan[0][0]}")

Contoh Skor Output Model AI (Scaled) : 0.75
Skor Dikembalikan ke Skala Asli (1-5)  : 4.0


# MENYIMPAN CLEANED DATASET

In [20]:
df.to_csv(
    'cleaned_sample_data.csv',
    index=False
)

# KESIMPULAN DATA CLEANING:

1. Missing value berhasil dicek.
2. Tidak ada duplicate data.
3. Dilakukan filtering active users.
4. Dilakukan filtering popular products.
5. Timestamp berhasil dikonversi ke format datetime.
6. Dataset siap digunakan untuk tahap EDA dan modeling.
7. Kolom `rating` telah dinormalisasi menggunakan MinMaxScaler ke rentang 0-1 untuk menyesuaikan arsitektur Deep Learning (Sigmoid Output).
8. Objek scaler telah di-export menjadi `rating_scaler.pkl` untuk kebutuhan inverse transform pada API dan Dashboard.